# Aprendizaje por refuerzo

In [ ]:
import numpy as np
# Definir el entorno (matriz de recompensas)
# -1: pared, 0: camino, 1: objetivo
maze = np.array([
 [ 0, -1, 0, 0, 1],
 [ 0, -1, 0,-1,-1],
 [ 0, 0, 0,  0, 0],
 [-1,-1, -1, 0, 0]
])
# Definir parámetros del entorno
num_states = maze.size
num_actions = 4 # Arriba, Abajo, Izquierda, Derecha

In [ ]:
# Definir una función de recompensa
def get_reward(state):
  row, col = state
  if maze[row, col] == 1:
    return 10 # Llegar a la meta
  if maze[row, col] == -1:
    return -10 # Chocar contra una pared
  return -1 # Cualquier otro movimiento

 # Función para realizar una acción y moverse en el entorno
def take_action(state, action):
 row, col = state
 if action == 0 and row > 0: # Arriba
    row -= 1
 elif action == 1 and row < maze.shape[0] - 1: # Abajo
    row += 1
 elif action == 2 and col > 0: # Izquierda
    col -= 1
 elif action == 3 and col < maze.shape[1] - 1: # Derecha
    col += 1
 return (row, col)

In [ ]:
# Inicializar tabla Q con ceros
Q_table = np.zeros((maze.shape[0], maze.shape[1], num_actions))

# Parámetros del Q-Learning
alpha = 0.1 # Tasa de aprendizaje
gamma = 0.9 # Descuento futuro
epsilon = 0.1 # Exploración vs explotación

# Función Q-Learning
def q_learning(epochs=1000):
  for _ in range(epochs):
    state = (np.random.randint(maze.shape[0]), np.random.randint(maze.shape[1]))
    while maze[state] != 1:
      action = np.argmax(Q_table[state]) if np.random.uniform(0, 1) > epsilon else np.random.randint(0, num_actions)
      new_state = take_action(state, action)
      reward = get_reward(new_state)
      best_future_q = np.max(Q_table[new_state])
      Q_table[state][action] += alpha * (reward + gamma * best_future_q - Q_table[state][action])
      state = new_state
# Entrenamos el modelo
q_learning()

In [ ]:
# Función para ejecutar el agente entrenado
def test_agent(start):
  state = start
  path = [state]
  while maze[state] != 1:
    action = np.argmax(Q_table[state])
    state = take_action(state, action)
    path.append(state)
  return path

# Evaluar el agente desde una posición de inicio
start_state = (0, 0)
path = test_agent(start_state)
print("Camino recorrido:", path)



Camino recorrido: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (1, 2), (0, 2), (0, 3), (0, 4)]


In [ ]:
!pip install keras gym numpy tensorflow
import gym
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10

# Cargar el dataset de CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
# Normalizar los datos
x_train = x_train / 255.0
x_test = x_test / 255.0


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [ ]:
def create_q_network(input_shape, num_actions):
    model = tf.keras.models.Sequential()
    model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dense(num_actions, activation='linear'))
    return model

# Definir los parámetros
input_shape = x_train.shape[1:] # Dimensiones de la imagen
num_actions = 10 # CIFAR-10 tiene 10 clases

# Crear la red Q
q_network = create_q_network(input_shape, num_actions)
q_network.compile(optimizer='adam', loss='mse')

In [ ]:
from collections import deque
import random

# Parámetros de DQN
epsilon_decay = 0.995
min_epsilon = 0.01
gamma = 0.9
batch_size = 32
memory = deque(maxlen=2000)

# Función de entrenamiento
def train_dqn(epochs=1000):
  for epoch in range(epochs):
  # Seleccionar una acción aleatoria o basada en la red Q
    if np.random.rand() <= epsilon:
      action = random.choice(range(num_actions))
    else:
      q_values = q_network.predict(x_train)
      action = np.argmax(q_values[0])

      # Almacenar la experiencia en el buffer
      memory.append((x_train, action, reward, x_train, done))

      if len(memory) > batch_size:
        batch = random.sample(memory, batch_size)
        for state, action, reward, next_state, done in batch:
            target = reward
            if not done:
                target = reward + gamma * np.max(q_network.predict(next_state)[0])
            target_f = q_network.predict(state)
            target_f[0][action] = target
            q_network.fit(state, target_f, epochs=1, verbose=0)

In [ ]:
def evaluate_model(x_test, y_test):
    score = q_network.evaluate(x_test, y_test, verbose=0)
    print(f"Precisión del agente en el test: {score*100:.2f}%")

# Evaluar el modelo entrenado
evaluate_model(x_test, y_test)

Precisión del agente en el test: 2846.59%
